In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------
PROJECT_DIR = Path("/Users/vedikabaradwaj/Documents/pdi_pensions")
DATA_DIR    = PROJECT_DIR / "regression_analysis" / "analytic samples" / "recruitment"

OUTPUT_DIR  = Path.home() / "Documents"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_PAIRS = [
    (2008, 2009),
    (2009, 2010),
    (2010, 2011),
    (2011, 2012),
    (2012, 2013),
    (2013, 2014),
    (2014, 2015),
    (2015, 2016),
    (2016, 2017),
]

FILE_PATTERN = "analytic_sample_{yy0}{yy1}_recruitment.csv"

DV            = "joined_state_local"
TREATMENT_VAR = "illinois"
PAIR_VAR      = "pair"

WEIGHT_CANDIDATES = [
    "weight_t", "wgt_t", "wtfinl_t", "finalwgt_t",
    "weight", "wgt", "wtfinl", "finalwgt",
    "earnwt_t", "earnwt"
]

COV_TYPE = "HC1"

EXCLUDED_STATE_FIPS  = {42}
EXCLUDED_STATE_NAMES = {"pa", "pennsylvania"}

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists?", DATA_DIR.exists())
if DATA_DIR.exists():
    print("CSV files found:")
    print([p.name for p in sorted(DATA_DIR.glob("*.csv"))])

# -------------------------------------------------------------------
# CONTROL VARIABLES
# -------------------------------------------------------------------
CONTINUOUS_CONTROL_CANDIDATES = [
    "age_t",
    "I(age_t**2)",
    "np.log(earnwke_t)",
]

CATEGORICAL_CONTROL_CANDIDATES = [
    "C(sex_t)",
    "C(race_t)",
    "C(ethnic_t)",
    "C(ethnicity_t)",
    "C(hispan_t)",
    "C(hispanic_t)",
    "C(hisp_t)",
    "C(grade92_t)",
    "C(educ_t)",
    "C(docc00_t)",
    "C(docc80_t)",
    "C(occ2010_t)",
    "C(occ_t)",
    "C(ind02_t)",
    "C(naics2_t)",
    "C(ind_t)",
    "C(unionmme_t)",
    "C(unioncov_t)",
]

ALL_CONTROL_CANDIDATES = CONTINUOUS_CONTROL_CANDIDATES + CATEGORICAL_CONTROL_CANDIDATES


def extract_needed_columns(terms):
    cols = set()
    for term in terms:
        for m in re.findall(r"C\(([^)]+)\)", term):
            cols.add(m.strip())
        for m in re.findall(r"np\.log\(([^)]+)\)", term):
            cols.add(m.strip())
        for expr in re.findall(r"I\(([^)]+)\)", term):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
        if not term.startswith(("C(", "I(", "np.log(")):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", term):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
    return sorted(cols)


def is_categorical_term(term):
    return term.startswith("C(")


def categorical_column_from_term(term):
    m = re.match(r"C\(([^)]+)\)", term)
    return m.group(1).strip() if m else None


def choose_one_present(terms, priority):
    present = [t for t in priority if t in terms]
    if len(present) <= 1:
        return terms
    keep = present[0]
    return [t for t in terms if (t not in priority or t == keep)]


def available_terms(df, candidate_terms):
    terms = []
    for term in candidate_terms:
        raw_cols = extract_needed_columns([term])
        if all(c in df.columns for c in raw_cols):
            terms.append(term)
    terms = choose_one_present(terms, ["C(docc00_t)", "C(docc80_t)", "C(occ2010_t)", "C(occ_t)"])
    terms = choose_one_present(terms, ["C(ind02_t)", "C(naics2_t)", "C(ind_t)"])
    terms = choose_one_present(terms, ["C(hispan_t)", "C(hispanic_t)", "C(hisp_t)", "C(ethnic_t)", "C(ethnicity_t)"])
    return terms


def build_formula(dv, controls):
    rhs_terms = [TREATMENT_VAR] + controls
    return f"{dv} ~ " + " + ".join(rhs_terms)


def detect_weight_var(df):
    for c in WEIGHT_CANDIDATES:
        if c in df.columns:
            return c
    return None


DATA_DIR: /Users/vedikabaradwaj/Documents/pdi_pensions/regression_analysis/analytic samples/recruitment
DATA_DIR exists? True
CSV files found:
['analytic_sample_0809_recruitment.csv', 'analytic_sample_0910_recruitment.csv', 'analytic_sample_1011_recruitment.csv', 'analytic_sample_1112_recruitment.csv', 'analytic_sample_1213_recruitment.csv', 'analytic_sample_1314_recruitment.csv', 'analytic_sample_1415_recruitment.csv', 'analytic_sample_1516_recruitment.csv', 'analytic_sample_1617_recruitment.csv']


In [13]:

# -------------------------------------------------------------------
# FILE LOADING + NY REMOVAL
# -------------------------------------------------------------------

def pair_to_file(y0, y1):
    yy0 = str(y0)[-2:]
    yy1 = str(y1)[-2:]
    return DATA_DIR / FILE_PATTERN.format(yy0=yy0, yy1=yy1)


def remove_ny(df):
    before = len(df)
    mask   = pd.Series(False, index=df.index)
    used_cols = []

    # Numeric FIPS columns — use stfips_t pre-2013, stfips post-2013
    for col in ["stfips_t", "stfips"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.isin(EXCLUDED_STATE_FIPS)
            used_cols.append(col)

    # String state name columns
    for col in ["state_name_t", "state_abbrev_t", "state_t"]:
        if col in df.columns:
            vals = df[col].astype(str).str.strip().str.lower()
            mask = mask | vals.isin(EXCLUDED_STATE_NAMES) | vals.str.contains("new york", na=False)
            used_cols.append(col)

    # NYC CBSA code
    for col in ["cbsafips_t", "cbsa_t"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.eq(35620)
            used_cols.append(col)

    df2 = df.loc[~mask].copy()
    print(f"Removed NY rows: {before - len(df2):,} using columns {sorted(set(used_cols))}")
    return df2


def load_pair(y0, y1):
    path = pair_to_file(y0, y1)
    print("\n" + "=" * 90)
    print(f"PAIR {y0}-{y1}  |  Looking for: {path.name}")

    if not path.exists():
        print(f"WARNING: file not found, skipping.")
        return None

    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
    df[PAIR_VAR] = f"{str(y0)[-2:]}{str(y1)[-2:]}"

    print(f"Loaded shape: {df.shape}")
    df = remove_ny(df)
    print(f"Shape after NY removal: {df.shape}")
    print(f"Detected weight variable: {detect_weight_var(df)}")
    return df

# -------------------------------------------------------------------
# DATA CLEANING
# -------------------------------------------------------------------

def clean_categorical_series(s):
    out = s.astype("object")
    out = out.where(~pd.isna(out), "MISSING")
    out = out.astype(str).str.strip()
    out = out.replace({
        "": "MISSING", "<NA>": "MISSING", "nan": "MISSING",
        "NaN": "MISSING", "None": "MISSING", "none": "MISSING",
    })
    return out.astype("object")


def prepare_model_data(df, controls):
    weight_var = detect_weight_var(df)
    if weight_var is None:
        return None, None, "No weight variable found"

    missing = [c for c in [DV, TREATMENT_VAR] if c not in df.columns]
    if missing:
        return None, None, f"Missing required columns: {missing}"

    data = df.copy()
    data[weight_var]    = pd.to_numeric(data[weight_var],    errors="coerce")
    data[DV]            = pd.to_numeric(data[DV],            errors="coerce")
    data[TREATMENT_VAR] = pd.to_numeric(data[TREATMENT_VAR], errors="coerce")

    numeric_cols = [DV, TREATMENT_VAR, weight_var]
    for term in controls:
        if not is_categorical_term(term):
            numeric_cols.extend(extract_needed_columns([term]))
    numeric_cols = sorted(set(c for c in numeric_cols if c in data.columns))

    for c in numeric_cols:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    for term in controls:
        if is_categorical_term(term):
            c = categorical_column_from_term(term)
            if c and c in data.columns:
                data[c] = clean_categorical_series(data[c])

    if "np.log(earnwke_t)" in controls and "earnwke_t" in data.columns:
        data = data[data["earnwke_t"] > 0].copy()

    before = len(data)
    data = data.dropna(subset=numeric_cols).copy()
    data = data[data[weight_var] > 0].copy()
    after = len(data)

    if data.empty:
        return None, weight_var, "No rows left after cleaning"

    return data, weight_var, None

# -------------------------------------------------------------------
# REGRESSION — ALL COEFFICIENTS
# -------------------------------------------------------------------

def classify_term(term):
    term_l = term.lower()
    if term == "Intercept":
        return "intercept"
    if term == TREATMENT_VAR:
        return "treatment"
    if "race" in term_l:
        return "race"
    if "sex" in term_l:
        return "sex"
    if any(x in term_l for x in ["hisp", "ethnic", "ethnicity"]):
        return "ethnicity_hispanic"
    if any(x in term_l for x in ["docc", "occ"]):
        return "occupation"
    if "grade" in term_l or "educ" in term_l:
        return "education"
    if "ind" in term_l or "naics" in term_l:
        return "industry"
    if "union" in term_l:
        return "union"
    if "age" in term_l:
        return "age"
    if "earn" in term_l or "log" in term_l:
        return "earnings"
    return "other"


def run_weighted_lpm_all_terms(df, y0, y1):
    controls = available_terms(df, ALL_CONTROL_CANDIDATES)
    formula  = build_formula(DV, controls)

    data, weight_var, error = prepare_model_data(df, controls)
    if error:
        print(f"Skipping {y0}-{y1}: {error}")
        return []

    print(f"\nRunning weighted full-covariate LPM for {y0}-{y1}")
    print("Formula:", formula)
    print(f"Rows used: {len(data):,} / {len(df):,}")
    print("Weight variable:", weight_var)

    try:
        result = smf.wls(
            formula=formula,
            data=data,
            weights=data[weight_var]
        ).fit(cov_type=COV_TYPE)
    except Exception as e:
        print(f"Regression failed for {y0}-{y1}: {e}")
        return []

    conf_int = result.conf_int()
    rows = []

    for term in result.params.index:
        coef     = result.params.get(term, np.nan)
        se       = result.bse.get(term, np.nan)
        pval     = result.pvalues.get(term, np.nan)
        ci_low   = conf_int.loc[term, 0] if term in conf_int.index else np.nan
        ci_high  = conf_int.loc[term, 1] if term in conf_int.index else np.nan

        rows.append({
            "pair":                   f"{y0}-{y1}",
            "pair_code":              f"{str(y0)[-2:]}{str(y1)[-2:]}",
            "dv":                     DV,
            "term":                   term,
            "term_group":             classify_term(term),
            "coef":                   coef,
            "coef_pct_points":        coef    * 100 if pd.notna(coef)    else np.nan,
            "std_err":                se,
            "std_err_pct_points":     se      * 100 if pd.notna(se)      else np.nan,
            "p_value":                pval,
            "ci_low":                 ci_low,
            "ci_high":                ci_high,
            "ci_low_pct_points":      ci_low  * 100 if pd.notna(ci_low)  else np.nan,
            "ci_high_pct_points":     ci_high * 100 if pd.notna(ci_high) else np.nan,
            "nobs":                   int(result.nobs),
            "r_squared":              result.rsquared,
            "weight_var":             weight_var,
            "controls":               ", ".join(controls),
            "file":                   pair_to_file(y0, y1).name,
        })

    return rows

# -------------------------------------------------------------------
# RUN ALL YEAR PAIRS
# -------------------------------------------------------------------

all_coef_rows = []
loaded_pairs  = {}

for y0, y1 in YEAR_PAIRS:
    df_pair = load_pair(y0, y1)
    if df_pair is None:
        continue

    loaded_pairs[f"{y0}-{y1}"] = df_pair
    rows = run_weighted_lpm_all_terms(df_pair, y0, y1)
    all_coef_rows.extend(rows)

all_results_table = pd.DataFrame(all_coef_rows)

print("\n" + "=" * 90)
print("DONE")
print(f"Coefficient rows produced: {len(all_results_table):,}")
print(f"Year pairs successfully loaded: {list(loaded_pairs.keys())}")
display(all_results_table.head(20))

# -------------------------------------------------------------------
# TABLE 1: ILLINOIS EFFECT BY YEAR PAIR
# -------------------------------------------------------------------

illinois_effect_table = (
    all_results_table
    .query("term_group == 'treatment'")
    [["pair", "pair_code", "term", "coef", "coef_pct_points", "std_err", "std_err_pct_points",
      "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var", "file"]]
    .sort_values("pair_code")
    .reset_index(drop=True)
)
display(illinois_effect_table)

# -------------------------------------------------------------------
# TABLE 2: ALL COVARIATE EFFECTS
# -------------------------------------------------------------------

all_covariate_effects_table = (
    all_results_table
    .query("term != 'Intercept'")
    [["pair", "pair_code", "term_group", "term", "coef", "coef_pct_points", "std_err",
      "std_err_pct_points", "p_value", "ci_low_pct_points", "ci_high_pct_points",
      "nobs", "r_squared", "weight_var"]]
    .sort_values(["pair_code", "term_group", "term"])
    .reset_index(drop=True)
)
display(all_covariate_effects_table)

# -------------------------------------------------------------------
# TABLE 3: DEMOGRAPHIC, OCCUPATION, INDUSTRY, UNION, AGE, EARNINGS
# -------------------------------------------------------------------

focus_groups = [
    "treatment", "race", "sex", "ethnicity_hispanic",
    "occupation", "education", "industry", "union", "age", "earnings"
]

focus_effects_table = (
    all_results_table
    .query("term_group in @focus_groups and term != 'Intercept'")
    [["pair", "pair_code", "term_group", "term", "coef_pct_points", "std_err_pct_points",
      "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var"]]
    .sort_values(["pair_code", "term_group", "term"])
    .reset_index(drop=True)
)
display(focus_effects_table)

# -------------------------------------------------------------------
# TABLE 4: AVERAGE COEFFICIENT ACROSS YEAR PAIRS BY TERM
# -------------------------------------------------------------------

average_effects_by_term = (
    all_results_table
    .query("term != 'Intercept'")
    .groupby(["term_group", "term"], as_index=False)
    .agg(
        mean_coef=("coef", "mean"),
        mean_coef_pct_points=("coef_pct_points", "mean"),
        median_coef_pct_points=("coef_pct_points", "median"),
        num_year_pairs=("pair_code", "nunique"),
        mean_nobs=("nobs", "mean"),
    )
    .sort_values(["term_group", "term"])
    .reset_index(drop=True)
)
display(average_effects_by_term)

# -------------------------------------------------------------------
# SAVE
# -------------------------------------------------------------------

out_all_csv      = OUTPUT_DIR / "recruitment_weighted_all_covariate_coefficients.csv"
out_illinois_csv = OUTPUT_DIR / "recruitment_weighted_illinois_effects.csv"
out_focus_csv    = OUTPUT_DIR / "recruitment_weighted_demographic_occupation_effects.csv"
out_avg_csv      = OUTPUT_DIR / "recruitment_weighted_average_effects_by_term.csv"
out_xlsx         = OUTPUT_DIR / "recruitment_weighted_regression_tables.xlsx"

all_results_table.to_csv(out_all_csv, index=False)
illinois_effect_table.to_csv(out_illinois_csv, index=False)
focus_effects_table.to_csv(out_focus_csv, index=False)
average_effects_by_term.to_csv(out_avg_csv, index=False)

with pd.ExcelWriter(out_xlsx) as writer:
    illinois_effect_table.to_excel(writer,        sheet_name="illinois_effects",    index=False)
    all_covariate_effects_table.to_excel(writer,  sheet_name="all_covariates",      index=False)
    focus_effects_table.to_excel(writer,          sheet_name="demo_occ_effects",    index=False)
    average_effects_by_term.to_excel(writer,      sheet_name="avg_by_term",         index=False)
    all_results_table.to_excel(writer,            sheet_name="raw_all_terms",       index=False)

print("Saved:")
print("-", out_all_csv)
print("-", out_illinois_csv)
print("-", out_focus_csv)
print("-", out_avg_csv)
print("-", out_xlsx)


PAIR 2008-2009  |  Looking for: analytic_sample_0809_recruitment.csv
Loaded shape: (18302, 205)
Removed NY rows: 8,859 using columns ['cbsafips_t', 'state_t', 'stfips_t']
Shape after NY removal: (9443, 205)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2008-2009
Formula: joined_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(unionmme_t) + C(unioncov_t)
Rows used: 4,842 / 9,443
Weight variable: weight_t

PAIR 2009-2010  |  Looking for: analytic_sample_0910_recruitment.csv
Loaded shape: (18724, 203)
Removed NY rows: 8,993 using columns ['cbsafips_t', 'state_t', 'stfips_t']
Shape after NY removal: (9731, 203)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2009-2010
Formula: joined_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(union

,pair,pair_code,dv,term,term_group,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low,ci_high,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,controls,file
0,2008-2009,0809,joined_state_local,Intercept,intercept,0.009885,0.988484,0.028371,2.837069,0.727527,-0.045721,0.065490,-4.572069,6.549036,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
1,2008-2009,0809,joined_state_local,C(sex_t)[T.2],sex,0.012330,1.233000,0.007035,0.703523,0.079669,-0.001459,0.026119,-0.145880,2.611881,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
2,2008-2009,0809,joined_state_local,C(race_t)[T.10],race,-0.008610,-0.860976,0.038042,3.804210,0.820951,-0.083171,0.065951,-8.317090,6.595138,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
3,2008-2009,0809,joined_state_local,C(race_t)[T.13],race,-0.052173,-5.217345,0.037759,3.775913,0.167050,-0.126180,0.021833,-12.617999,2.183309,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
4,2008-2009,0809,joined_state_local,C(race_t)[T.15],race,0.002180,0.217985,0.022243,2.224250,0.921929,-0.041415,0.045774,-4.141466,4.577435,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
5,2008-2009,0809,joined_state_local,C(race_t)[T.2],race,0.007933,0.793252,0.013301,1.330091,0.550915,-0.018137,0.034002,-1.813679,3.400183,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
6,2008-2009,0809,joined_state_local,C(race_t)[T.3],race,0.066312,6.631201,0.080579,8.057860,0.410537,-0.091619,0.224243,-9.161914,22.424316,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
7,2008-2009,0809,joined_state_local,C(race_t)[T.4],race,-0.029042,-2.904227,0.014475,1.447478,0.044813,-0.057412,-0.000672,-5.741232,-0.067221,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
8,2008-2009,0809,joined_state_local,C(race_t)[T.5],race,-0.053129,-5.312874,0.046387,4.638700,0.252070,-0.144046,0.037788,-14.404559,3.778810,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
9,2008-2009,0809,joined_state_local,C(race_t)[T.6],race,-0.006635,-0.663454,0.021159,2.115897,0.753858,-0.048105,0.034836,-4.810536,3.483629,4842,0.10208,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv


,pair,pair_code,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,file
0,2008-2009,0809,illinois,-0.009047,-0.904704,0.006200,0.620012,0.144518,-2.119905,0.310496,4842,0.102080,weight_t,analytic_sample_0809_recruitment.csv
1,2009-2010,0910,illinois,-0.003820,-0.382007,0.006105,0.610518,0.531505,-1.578600,0.814586,4816,0.108873,weight_t,analytic_sample_0910_recruitment.csv
2,2010-2011,1011,illinois,-0.003637,-0.363669,0.006261,0.626068,0.561323,-1.590740,0.863402,4716,0.086663,weight_t,analytic_sample_1011_recruitment.csv
3,2011-2012,1112,illinois,-0.008314,-0.831424,0.006094,0.609372,0.172443,-2.025771,0.362923,4599,0.095493,weight_t,analytic_sample_1112_recruitment.csv
4,2012-2013,1213,illinois,0.007239,0.723862,0.006219,0.621937,0.244471,-0.495112,1.942837,4616,0.105457,weight_t,analytic_sample_1213_recruitment.csv
5,2013-2014,1314,illinois,-0.002333,-0.233326,0.006022,0.602247,0.698441,-1.413708,0.947057,3960,0.100381,weight_t,analytic_sample_1314_recruitment.csv
6,2014-2015,1415,illinois,0.001490,0.149017,0.006443,0.644347,0.817107,-1.113880,1.411914,2955,0.191698,weight_t,analytic_sample_1415_recruitment.csv
7,2015-2016,1516,illinois,-0.020470,-2.046957,0.006237,0.623708,0.001031,-3.269402,-0.824512,3332,0.194056,weight_t,analytic_sample_1516_recruitment.csv
8,2016-2017,1617,illinois,-0.004175,-0.417512,0.006118,0.611762,0.494939,-1.616543,0.781520,3312,0.157600,weight_t,analytic_sample_1617_recruitment.csv


,pair,pair_code,term_group,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.000018,-0.001781,0.000015,0.001530,0.244478,-0.004780,0.001218,4842,0.10208,weight_t
1,2008-2009,0809,age,age_t,0.001965,0.196470,0.001352,0.135240,0.146293,-0.068596,0.461535,4842,0.10208,weight_t
2,2008-2009,0809,earnings,np.log(earnwke_t),-0.007792,-0.779208,0.005578,0.557806,0.162439,-1.872489,0.314072,4842,0.10208,weight_t
3,2008-2009,0809,education,C(grade92_t)[T.32],-0.053123,-5.312311,0.025745,2.574541,0.039075,-10.358318,-0.266303,4842,0.10208,weight_t
4,2008-2009,0809,education,C(grade92_t)[T.33],-0.059374,-5.937382,0.023041,2.304081,0.009969,-10.453299,-1.421466,4842,0.10208,weight_t
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2640,2016-2017,1617,sex,C(sex_t)[T.2],0.008164,0.816437,0.006611,0.661058,0.216813,-0.479212,2.112087,3312,0.15760,weight_t
2641,2016-2017,1617,treatment,illinois,-0.004175,-0.417512,0.006118,0.611762,0.494939,-1.616543,0.781520,3312,0.15760,weight_t
2642,2016-2017,1617,union,C(unioncov_t)[T.2.0],-0.029581,-2.958120,0.043637,4.363698,0.497840,-11.510811,5.594572,3312,0.15760,weight_t
2643,2016-2017,1617,union,C(unioncov_t)[T.MISSING],-0.002794,-0.279382,0.022299,2.229880,0.900294,-4.649867,4.091103,3312,0.15760,weight_t


,pair,pair_code,term_group,term,coef_pct_points,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.001781,0.001530,0.244478,-0.004780,0.001218,4842,0.10208,weight_t
1,2008-2009,0809,age,age_t,0.196470,0.135240,0.146293,-0.068596,0.461535,4842,0.10208,weight_t
2,2008-2009,0809,earnings,np.log(earnwke_t),-0.779208,0.557806,0.162439,-1.872489,0.314072,4842,0.10208,weight_t
3,2008-2009,0809,education,C(grade92_t)[T.32],-5.312311,2.574541,0.039075,-10.358318,-0.266303,4842,0.10208,weight_t
4,2008-2009,0809,education,C(grade92_t)[T.33],-5.937382,2.304081,0.009969,-10.453299,-1.421466,4842,0.10208,weight_t
...,...,...,...,...,...,...,...,...,...,...,...,...
2640,2016-2017,1617,sex,C(sex_t)[T.2],0.816437,0.661058,0.216813,-0.479212,2.112087,3312,0.15760,weight_t
2641,2016-2017,1617,treatment,illinois,-0.417512,0.611762,0.494939,-1.616543,0.781520,3312,0.15760,weight_t
2642,2016-2017,1617,union,C(unioncov_t)[T.2.0],-2.958120,4.363698,0.497840,-11.510811,5.594572,3312,0.15760,weight_t
2643,2016-2017,1617,union,C(unioncov_t)[T.MISSING],-0.279382,2.229880,0.900294,-4.649867,4.091103,3312,0.15760,weight_t


,term_group,term,mean_coef,mean_coef_pct_points,median_coef_pct_points,num_year_pairs,mean_nobs
0,age,I(age_t ** 2),-0.000003,-0.000308,-0.000292,9,4127.555556
1,age,age_t,0.000286,0.028554,0.015926,9,4127.555556
2,earnings,np.log(earnwke_t),-0.005004,-0.500371,-0.719995,9,4127.555556
3,education,C(grade92_t)[T.32],0.000175,0.017472,-0.047094,9,4127.555556
4,education,C(grade92_t)[T.33],0.003410,0.340973,-0.475374,9,4127.555556
...,...,...,...,...,...,...,...
329,sex,C(sex_t)[T.2],0.006812,0.681164,0.751527,9,4127.555556
330,treatment,illinois,-0.004785,-0.478524,-0.382007,9,4127.555556
331,union,C(unioncov_t)[T.2.0],-0.008664,-0.866441,-1.723857,9,4127.555556
332,union,C(unioncov_t)[T.MISSING],0.014173,1.417346,3.693108,9,4127.555556


Saved:
- /Users/vedikabaradwaj/Downloads/recruitment_weighted_all_covariate_coefficients.csv
- /Users/vedikabaradwaj/Downloads/recruitment_weighted_illinois_effects.csv
- /Users/vedikabaradwaj/Downloads/recruitment_weighted_demographic_occupation_effects.csv
- /Users/vedikabaradwaj/Downloads/recruitment_weighted_average_effects_by_term.csv
- /Users/vedikabaradwaj/Downloads/recruitment_weighted_regression_tables.xlsx
